# Final Flood Score

For each subdistrict $s$, the flood index for a given day $d$ is $$ \phi_{i,d} := \math{1}{pr_{i,d}\geq\tau_i} \cdot (1 + SM_{anomaly,i,d})$$ where $pr_d$ is the precipitation on subdistrict $i$ District-level values for each day are obtained by taking the maximum across subdistricts --- reflecting the worst-hit subdistrict within the district --- and the yearly district-level index is defined as the sum of these values over the hydrological year (July--June): $$F_{\delta,y} = \sum_{d \in y} \max_{i \in \Delta}\phi_{i,d}$$ \noindent where $\Delta$ is the set of subdistricts in district $\delta$, and $y$ is the set of days within hydrological year $y$. Finally, population-weighted averages across districts yield the state-level yearly flood index. $$FI_{state, y} = \frac{\sum_{\delta \in state} F_{\delta, y} \cdot pop_\delta}{\sum_\delta pop_\delta}$$ In essence, the state-level index captures the frequency of extreme precipitation events weighted by antecedent soil moisture conditions, with each event contributing its most severe local manifestation. The emphasis is on event frequency rather than intensity --- the threshold is calibrated to capture only severe precipitation, so soil moisture modulation plays a secondary role. Whether this weighting meaningfully improves upon a raw event count will be assessed empirically in the regression analysis.

In [42]:
import pandas as pd
import numpy as np
from glob import glob
import re
import unicodedata

from fontTools.diff import color
from pandas.core.array_algos import replace

### Load data

In [43]:
def reshape_precip(file_path):
    df = pd.read_csv(file_path)

    # 1. Full list of metadata columns you *want*
    desired_meta = ['OBJECTID_1', 'REMARKS', 'STATE_LGD', 'STATE_UT', 'DIST_LGD',
                    'SUBDIS_LGD', 'SUBDIS_TYP', 'SUB_DIST', 'system:index',
                    'Shape_Area',  'Shape_Leng',  '.geo', 'DISTRICT', 'OBJECTID']

    # 2. Only use the columns that actually exist in THIS specific file
    existing_meta = [c for c in desired_meta if c in df.columns]

    # 3. Date columns are everything that isn't in existing_meta
    date_cols = [c for c in df.columns if c not in existing_meta]

    # 4. Melt using only the columns we found
    df_long = df.melt(
        id_vars=existing_meta,
        value_vars=date_cols,
        var_name="date_raw",
        value_name="precip"
    )

    # Extract date
    df_long["date"] = pd.to_datetime(
        df_long["date_raw"].str.extract(r"(\d{8})")[0],
        format="%Y%m%d"
    )

    df_long = df_long.drop(columns=["date_raw"])

    return df_long

In [44]:
# -------------------------
# 1. Load data
# -------------------------

# thresholds (district-level)
thresholds = pd.read_csv("/Users/ninabilirossi/Desktop/MSC THESIS/Data works/Code/Outputs/thresholds_custom_year.csv")
 # district_id, tau

# soil moisture (full panel)
sm = pd.read_csv('/Users/ninabilirossi/Desktop/MSC THESIS/Data works/Code/Outputs/antecedent soil moisture/SM_Anomaly.csv', parse_dates=["date"])

In [45]:
# precipitation (multiple yearly files)
precip_files = glob("/Users/ninabilirossi/Desktop/MSC THESIS/Data works/Code/Data/daily max pr era5 2015-2024/*.csv")
#precip_files2 = glob('/Users/ninabilirossi/Desktop/MSC THESIS/Data works/Code/Data/daily max pr era5 missing states 2015-2024/*.csv')

all_files = precip_files # + precip_files2

precip = pd.concat(
    [reshape_precip(f) for f in all_files],
    ignore_index=True)

/var/folders/q2/z68hsy6j50x95qjg_gzyl4r00000gn/T/ipykernel_45218/856128151.py:2: DtypeWarning: Columns (0: DIST_LGD, 1: REMARKS, 2: SUBDIS_LGD) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)
/var/folders/q2/z68hsy6j50x95qjg_gzyl4r00000gn/T/ipykernel_45218/856128151.py:2: DtypeWarning: Columns (0: DIST_LGD, 1: REMARKS, 2: SUBDIS_LGD) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)
/var/folders/q2/z68hsy6j50x95qjg_gzyl4r00000gn/T/ipykernel_45218/856128151.py:2: DtypeWarning: Columns (0: DIST_LGD, 1: REMARKS, 2: SUBDIS_LGD) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)
/var/folders/q2/z68hsy6j50x95qjg_gzyl4r00000gn/T/ipykernel_45218/856128151.py:2: DtypeWarning: Columns (0: DIST_LGD, 1: REMARKS, 2: SUBDIS_LGD) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)
/var/fol

In [13]:
thresholds['STATE'].nunique()
#sm['SUB_DIST'].nunique()

33

In [46]:
# population
pop = pd.read_csv("~/Desktop/MSC THESIS/Data works/Code/Outputs/population.csv")  # district_id, population

### Process data

Before merging, check whether the district names match.

In [49]:
print(thresholds.columns)

Index(['STATE', 'DISTRICT', 'THRESHOLD_10YR'], dtype='str')


In [52]:
# What's in pop_df but not in combined_exceedance
print("Missing from precip:",
      set(thresholds['DISTRICT'].unique()) - set(precip['DISTRICT'].unique()))

# What's in combined_exceedance but not in merged_df
print("Missing from thresholds:",
      set(precip['DISTRICT'].unique()) - set(thresholds['DISTRICT'].unique()))

Missing from precip: set()
Missing from thresholds: {'SHAHDARA', 'NIULAND', 'NORTH EAST', 'KARAIKAL', 'DAMAN', nan, 'WEST', 'Lakshadweep'}


We see that a few very thin districts are missing (because they're too skinny for the satelite resolution) but that's okay.

In [54]:
# -------------------------
# 2. Merge pr and threshold to get events per subdistrict per date matrix (1 or 0)
# -------------------------
exceedance_df = precip.merge(
    thresholds,
    left_on=['STATE_UT', 'DISTRICT'],
    right_on=['STATE', 'DISTRICT'],
    how='inner'
)

In [55]:
exceedance_df['STATE_UT'].nunique()

33

In [56]:
# Binary exceedance (this is the index)
exceedance_df['exceedance'] = (
    exceedance_df['precip'] > exceedance_df['THRESHOLD_10YR']
).astype(int)

exceedance_df = exceedance_df.sort_values(
    by=['STATE_UT', 'DISTRICT', 'SUB_DIST', 'date']
)

exceedance_df['prev_exceedance'] = (
    exceedance_df
    .groupby(['STATE_UT', 'DISTRICT', 'SUB_DIST'])['exceedance']
    .shift(1)
    .fillna(0)
)

# New event starts when:
# today = 1 AND yesterday = 0
exceedance_df['new_event'] = (
    (exceedance_df['exceedance'] == 1) &
    (exceedance_df['prev_exceedance'] == 0)
)


In [24]:
print(exceedance_df['STATE_UT'].nunique())

33


Before merging, check whether the district names match.

In [57]:
print(exceedance_df.columns)

Index(['OBJECTID_1', 'REMARKS', 'STATE_LGD', 'STATE_UT', 'DIST_LGD',
       'SUBDIS_LGD', 'SUBDIS_TYP', 'SUB_DIST', 'system:index', 'Shape_Area',
       'Shape_Leng', '.geo', 'DISTRICT', 'OBJECTID', 'precip', 'date', 'STATE',
       'THRESHOLD_10YR', 'exceedance', 'prev_exceedance', 'new_event'],
      dtype='str')


In [58]:
# What's in pop_df but not in combined_exceedance
print("Missing from exceedance:",
      set(sm['SUB_DIST'].unique()) - set(exceedance_df['SUB_DIST'].unique()))

# What's in combined_exceedance but not in merged_df
print("Missing from sm:",
      set(exceedance_df['SUB_DIST'].unique()) - set(sm['SUB_DIST'].unique()))

Missing from exceedance: {'SHAHDARA', 'SEEMA PURI', 'DAMAN', 'NIHOKHU', 'Amini', 'Kalpeni', 'Kavaratti', 'NIULAND', 'RAJOURI GARDEN', 'YAMUNA VIHAR', 'KARAWAL NAGAR', 'Chetlat', 'Bitra', 'PUNJABI BAGH', 'KARAIKAL', 'PATEL NAGAR', 'Agatti', 'Kadmat', 'Andrott', 'Kiltan', 'Minicoy', 'VIVEK VIHAR', 'SEELAMPUR', 'THIRUNALLAR'}
Missing from sm: set()


In [71]:
print(sm.columns)

Index(['SUB_DIST', 'year', 'date', 'DOY', 'real_val', 'clim_val', 'anomaly'], dtype='str')


In [59]:
# -------------------------
# 3. Compute φ_{i,d}
# -------------------------

df = exceedance_df.merge(sm,
    on=["SUB_DIST", "date"],
    how="left")
print(df.columns)

df["phi"] = df["new_event"] * (1 + df["anomaly"])

Index(['OBJECTID_1', 'REMARKS', 'STATE_LGD', 'STATE_UT', 'DIST_LGD',
       'SUBDIS_LGD', 'SUBDIS_TYP', 'SUB_DIST', 'system:index', 'Shape_Area',
       'Shape_Leng', '.geo', 'DISTRICT', 'OBJECTID', 'precip', 'date', 'STATE',
       'THRESHOLD_10YR', 'exceedance', 'prev_exceedance', 'new_event', 'year',
       'DOY', 'real_val', 'clim_val', 'anomaly'],
      dtype='str')


In [60]:
df['STATE'].nunique()

33

In [61]:
# -------------------------
# 4. District-day max over subdistricts
# -------------------------

district_day = (
    df.groupby(["DISTRICT", "date"])["phi"]
    .max()
    .reset_index()
)

In [62]:
# -------------------------
# 5. Define hydrological year (July–June)
# -------------------------

def hydro_year(date):
    if date.month >= 7:
        return date.year
    else:
        return date.year - 1

district_day["hydro_year"] = district_day["date"].apply(hydro_year)

In [63]:
# -------------------------
# 6. Compute F_{δ,y}
# -------------------------

district_year = (
    district_day.groupby(["DISTRICT", "hydro_year"])["phi"]
    .sum()
    .reset_index()
    .rename(columns={"phi": "F_dy"})
)

In [64]:
district_year['DISTRICT'].nunique()

731

In [33]:
# -------------------------
# 7. State-level weighted index
# -------------------------

In [ ]:
# Now we revert back to only the 33 states (excludee AP and M)
# Create sets of unique districts
'''
pop_districts = set(pop['DISTRICT'].unique())
exceedance_districts = set(exceedance_df['DISTRICT'].unique())

# Districts in 'pop' but NOT in 'exceedance_df'
missing_in_exceedance = pop_districts - exceedance_districts

# Districts in 'exceedance_df' but NOT in 'pop'
missing_in_pop = exceedance_districts - pop_districts

print(f"Missing in Exceedance: {missing_in_exceedance}")
print(f"Missing in Pop: {missing_in_pop}")
'''

In [65]:
def clean_name(x):
    if pd.isna(x):
        return x

    # Normalize unicode (fix hidden encoding issues)
    x = unicodedata.normalize("NFKD", x)

    # Remove non-ASCII characters
    x = x.encode("ascii", "ignore").decode("ascii")

    # Uppercase
    x = x.upper()

    # Remove special characters (like >, |, @)
    x = re.sub(r"[^A-Z0-9 ]", "", x)

    # Remove extra spaces
    x = re.sub(r"\s+", " ", x).strip()

    return x

In [66]:
district_year["DISTRICT_CLEAN"] = district_year["DISTRICT"].apply(clean_name)
pop["DISTRICT_CLEAN"] = pop["DISTRICT"].apply(clean_name)

In [67]:
# OPTIONAL, might create errors
from rapidfuzz import process

choices = pop["DISTRICT_CLEAN"].unique()

def fuzzy_match(x):
    match, score, _ = process.extractOne(x, choices)
    return match if score > 90 else None

district_year["DISTRICT_MATCH"] = district_year["DISTRICT_CLEAN"].apply(fuzzy_match)

In [69]:
print(district_year['DISTRICT_CLEAN'].nunique())
print(pop['DISTRICT_CLEAN'].nunique())

731
778


In [70]:
# Create sets of unique districts
pop_districts = set(pop['DISTRICT_CLEAN'].unique())
dy_districts = set(district_year['DISTRICT_CLEAN'].unique())

# Districts in 'pop' but NOT in 'exceedance_df'
missing_in_exceedance = pop_districts - dy_districts

# Districts in 'exceedance_df' but NOT in 'pop'
missing_in_pop = dy_districts - pop_districts

print(f"Missing in District year: {missing_in_exceedance}")
print(f"Missing in Pop: {missing_in_pop}")

missing_after = set(pop['DISTRICT_CLEAN'].unique()) - set(district_year['DISTRICT_CLEAN'].unique())
print(f"Missing: {len(missing_after)}")


Missing in District year: {'EAST JAINTIA HILLS', 'KRA DAADI', 'WEST JAINTIA HILLS', 'KAMLE', 'MAMIT', 'KANGPOKPI', 'NANDURBAR', 'RIBHOI', 'BULDHANA', 'NJW', nan, 'SOUTH GRO HILLS', 'RAJAURI', 'ANANTNAG', 'PALGHAR', 'LATUR', 'KOLKATA', 'SIANG', 'TENGNOUPAL', 'RAIGAD', 'RUDRAPRAYG', 'KEYI PANYOR', 'NABARANGAPUR', 'BICHOM', 'BADGAM', 'MUMBAI SUBURBAN', 'MALER KOTLA', 'CHURACHANDPUR', 'NORTH GRO HILLS', 'BEED', 'JANJGIR CHAMPA', 'NAGPUR', 'UPPER SUBANSIRI', 'NAZUL', 'BHANDARA', 'LONGDING', 'SHAHDARA', 'SOUTH 24PARGANAS', 'SANGLI', 'LOWER SUBANSIRI', 'DAMAN', 'GANDERBAL', 'EAST KAMENG', 'RAMBAN', 'DARANG', 'PAURI GARHWL', 'NIULAND', 'WEST SIANG', 'SERCHHIP', 'TEHRI GARHWL', 'LEPA RADA', 'CHNGLNG', 'WEST KAMENG', 'SAMBA', 'THANE', 'TAWANG', 'CHAMPHAI', 'PAPUMPARE', 'SAS NAGAR SAHIBZADA AJIT SINGH NAGAR', 'SHI YOMI', 'NAMSAI', 'SOUTH WEST GRO HILLS', 'KAMJANG', 'DAKSHIN DINJPUR', 'NUAPARHA', 'EAST SIANG', 'CHAMPWAT', 'YAVATMAL', 'SRINAGAR', 'LOWER SIANG', 'KULGAM', 'SENAPATI', 'SOUTH WEST KHS

In [72]:
# Lookup: District year → Pop
district_lookup = {
    "KISHTWAR": "KISHTWR",
    "KULGAM": "KULGM",
    "DR BR AMBEDKAR KONASEEMA": "DRBRAMBEDKAR KONASEEMA",
    "NAINITL": "NAINITAL",
    "WASHIM": "WSHM",
    "PITHORGARH": "PITHORAGARH",
    "MALER KOTLA": "MALERKOTLA",
    "NANDURBAR": "NANDURBR",
    "RAJAURI": "RJAURI",
    "TEHRI GARHWL": "TEHRI GARHWAL",
    "CHAMPWAT": "CHAMPAWAT",
    "MURSHIDBD": "MURSHIDABAD",
    "PAURI GARHWL": "PAURI GARHWAL",
    "ANUGUL": "ANGUL",
    "TIRUVALLUR": "THIRUVALLUR",
    "DARANG": "DARRANG",
    "MUMBAI SUBURBAN": "SUB URBAN MUMBAI",
    "BGESHWAR": "BAGESHWAR",
    "RATNAGIRI": "RATNGIRI",
    "PURBA MEDINPUR": "PRBA MEDINPUR",
    "MUZAFFARABAD": "MUZAFFARBD",
    "SOLAPUR": "SOLPUR",
    "BARAMULA": "BRAMLA",
    "SOUTH 24PARGANAS": "SOUTH TWENTYFOUR PARGANAS",
    "MIRPUR": "MRPUR",
    "JALNA": "JLNA",
    "SAMBA": "SMBA",
    "DAKSHIN DINJPUR": "DAKSHIN DINAJPUR",
    "TENGNOUPAL": "TENGNOUPL",
    "THANE": "THNE",
    "ANANTNAG": "ANANTNG",
    "KEONJHAR KENDUJHAR": "KEONJHAR",
    "KANGPOKPI": "KNGPOKPI",
    "AMRAVATI": "AMARVATI",
    "NAGPUR": "NGPUR",
    "LAHUL SPITI": "LHUL SPITI",
    "RUDRAPRAYG": "RUDRAPRAYAG",
    "YSR": "YSR KADAPA",
    "RAMBAN": "RMBAN",
    "JAJAPUR": "JAJPUR",
    "JANJGIR CHAMPA": "JANJGR CHAMPA",
    "GARIYABAND": "GARIYABANDH",
    "NABARANGAPUR": "NABARANGPUR",
    "LATUR": "LTR",
    "KOLHAPUR": "KOLHPUR",
    "HARIDWR": "HARIDWAR",
    "CHAMPHAI": "CHAMPHI",
    "MAMIT": "MMIT",
    "YAVATMAL": "YAVATML",
    "SRI MUKTSAR SAHIB": "MUKTSAR",
    "PALGHAR": "PLGHAR",
    "PUNCH": "PNCH",
    "NUAPARHA": "NUAPADA",
    "BADGAM": "BADGM",
    "BHANDARA": "BHANDRA",
    "SERCHHIP": "SERCHHP",
    "SHUPIYAN": "SHUPYAN",
    "UTTARKSHI": "UTTARKASHI",
    "SENAPATI": "SENPATI",
    "SRINAGAR": "SRNAGAR",
    "BEED": "BD",
    "GANDERBAL": "GANDARBAL",
    "SAS NAGAR SAHIBZADA AJIT SINGH NAGAR": "SAHIBZADA AJIT SINGH NAGAR",
    "NASHIK": "NSHIK",
    "SANGLI": "SNGLI",
    "NANDED": "NNDED",
    "KAKCHING": "KKCHING",
    "CHURACHANDPUR": "CHRCHNDPUR",
    "RAIGAD": "RYGAD",
    "RIASI": "RISI",
    "KAMJANG": "KMJANG",
    "SATARA": "STRA",
    "BULDHANA": "BULDHNA",
    "THOUBAL": "THOUBL",
    "KARBI ANGLONG": "KARBI ANAGLONG",
    "AIZAWL": "ZWAL",
    "KOLKATA": "KOLKTA",
}

# Unmatched
missing_in_district_year = [
    "DAMAN", "LAKSHADWEEP", "KARAIKAL", "SHAHDARA",
    "NORTH EAST", "WEST", "NAZUL", "NIULAND"
]


In [74]:
# Apply the specific manual lookup table
# Ensure lookup keys are also uppercase for matching
lookup_upper = {k.upper(): v.upper() for k, v in district_lookup.items()}
district_year['DISTRICT_CLEAN'] = district_year['DISTRICT_CLEAN'].replace(lookup_upper)
pop['DISTRICT_CLEAN'] = pop['DISTRICT_CLEAN'].replace(lookup_upper)


# Final check
missing_after = set(pop['DISTRICT_CLEAN'].unique()) -set(district_year['DISTRICT_CLEAN'].unique())
print(f"Remaining missing: {len(missing_after)}")

print(missing_after)

Remaining missing: 48
{'EAST JAINTIA HILLS', 'KRA DAADI', 'WEST JAINTIA HILLS', 'SHAHDARA', 'KAMLE', 'SOUTH WEST KHSI HILLS', 'WEST KHASI HILLS', 'LOWER SUBANSIRI', 'RIBHOI', 'DAMAN', 'NJW', 'EAST KAMENG', nan, 'SOUTH GRO HILLS', 'WEST GRO HILLS', 'WEST', 'LOHIT', 'PAKKE KESSANG', 'NIULAND', 'WEST SIANG', 'LAKSHADWEEP', 'SIANG', 'KEYI PANYOR', 'BICHOM', 'UPPER SIANG', 'LOWER DIBNG VALLEY', 'LEPA RADA', 'CHNGLNG', 'WEST KAMENG', 'TIRP', 'EAST GRO HILLS', 'NORTH EAST', 'NORTH GRO HILLS', 'TAWANG', 'KARAIKAL', 'PAPUMPARE', 'SHI YOMI', 'NAMSAI', 'EAST KHSI HILLS', 'SOUTH WEST GRO HILLS', 'UPPER SUBANSIRI', 'EAST SIANG', 'EASTERN WEST KHASI HILLS', 'DIBNG VALLEY', 'KURUNG KUMEY', 'NAZUL', 'LONGDING', 'LOWER SIANG'}


In [75]:
#district_year = district_year.merge(pop, on="DISTRICT_CLEAN", how="left")

district_year = district_year.merge(
    pop,
    left_on="DISTRICT_CLEAN",
    right_on="DISTRICT_CLEAN",
    how="left"
)

In [76]:
district_year[district_year["population"].isna()]["DISTRICT_CLEAN"].unique()

<StringArray>
[]
Length: 0, dtype: str

In [77]:
district_year["DISTRICT"] = district_year["DISTRICT_CLEAN"]

In [78]:
district_year["weighted"] = district_year["F_dy"] * district_year["population"]

state_year = (
    district_year
    .groupby(["STATE_UT", "hydro_year"])
    .agg(
        FI_state=("weighted", "sum"),
        total_pop=("population", "sum")
    )
    .reset_index()
)

state_year["FI_state"] = state_year["FI_state"] / state_year["total_pop"]

state_year = state_year.drop(columns="total_pop")

In [79]:
district_year['STATE_UT'].unique()

<StringArray>
[                       'WEST BENGAL',                            'MIZORAM',
                          'TELANGANA',                     'MADHYA PRADESH',
                      'UTTAR PRADESH',                        'MAHARASHTRA',
                            'GUJARAT',                          'RAJASTHAN',
                     'ANDHRA PRADESH',                        'UTTAR>KHAND',
                            'HARYANA',                  'JAMMU AND KASHMIR',
                             'ODISHA',                         'TAMIL NADU',
                             'KERALA',                             'PUNJAB',
                              'BIHAR',                          'KARNATAKA',
                              'ASSAM',                       'CHHATTISGARH',
                   'HIMACHAL PRADESH',                            'MANIPUR',
                          'JHARKHAND',                              'DELHI',
                         'CHANDIGARH',                        

In [80]:
# a small test

subset = district_year[
    (district_year["STATE_UT"] == "MAHARASHTRA") &
    (district_year["hydro_year"] == 2018)
]

manual = (subset["F_dy"] * subset["population"]).sum() / subset["population"].sum()
print(manual)
print(state_year[
    (state_year["STATE_UT"] == "MAHARASHTRA") &
    (state_year["hydro_year"] == 2018)
])

0.11615985748436305
        STATE_UT  hydro_year  FI_state
221  MAHARASHTRA        2018   0.11616


In [81]:
# -------------------------
# 8. Output
# -------------------------

district_year.to_csv("/Users/ninabilirossi/Desktop/MSC THESIS/Data works/Code/Outputs/final material/district_flood_index.csv", index=False)
state_year.to_csv("/Users/ninabilirossi/Desktop/MSC THESIS/Data works/Code/Outputs/final material/state_flood_index.csv", index=False)

## Some plots for funsies

In [131]:
import geopandas as gpd
import matplotlib.pyplot as plt

shapefile_path = "~/Desktop/MSC THESIS/Data works/Code/Data/geography/State_District_Sub-district_Boundary_of_entire_India/District_Boundary.shp"

gdf = gpd.read_file(shapefile_path)

In [132]:
state_shapes = gdf.dissolve(by="STATE_UT", as_index=False)

In [ ]:
#print(gdf.columns)

In [133]:
state_year_map = (
    district_year
    .groupby(["STATE_UT", "hydro_year"])["F_dy"]
    .mean()   # or sum(), depending on your preference
    .reset_index()
)

In [134]:
years = sorted(state_year_map["hydro_year"].unique())

for year in years:
    data_year = state_year_map[state_year_map["hydro_year"] == year]

    # Merge with geometry
    map_df = state_shapes.merge(
        data_year,
        on="STATE_UT",
        how="left"
    )

    # Plot
    fig, ax = plt.subplots(1, 1, figsize=(10, 10))

    map_df.plot(
        column="F_dy",
        cmap="Reds",
        linewidth=0.5,
        edgecolor="black",
        legend=True,
        ax=ax,
        missing_kwds={"color": "lightgrey"}
    )
    legend = ax.get_legend()
    if legend:
        for text in legend.get_texts():
            text.set_color("black")
    ax.set_title(f"Flood Index by State ({year}-{year+1})", color="black")
    plt.rcParams["text.color"] = "black"
    ax.axis("off")

    plt.savefig(f"/Users/ninabilirossi/Desktop/MSC THESIS/Data works/Code/Outputs/final material/flood_index_map_{year}.png", dpi=300, bbox_inches="tight", facecolor="white")
    plt.close()